In [6]:
# 1. Data Cleaning
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("DataCleaning").master("local[*]").getOrCreate()

uncleanDF = spark.createDataFrame(
    [("Alice", 25, "NY"),
     ("Bob", None, "LA"),
     ("Bob", None, "LA"),
     (None, 40, "LA")],
    ["name", "age", "city"]
)

print("Uncleaned data:")
uncleanDF.show()

# Get rid of duplicate entries
noDuplicates = uncleanDF.dropDuplicates()
print("Removed duplicates:")
noDuplicates.show()

# Replace missing instances of age with average age
avg = noDuplicates.agg(func.avg(func.col("age"))).first()[0]
cleanedAge = noDuplicates.fillna(avg, subset=["age"])
print("Replaced missing ages with average age:")
cleanedAge.show()

# Replace missing names with "Unknown"
cleanedName = cleanedAge.fillna("Unknown", subset=["name"])
print("Replaced missing names with 'Unknown':")
cleanedName.show()

# Filter people over 30
aboveAge = cleanedName.where(func.col("age") > 30)
print("People over 30:")
aboveAge.show()

spark.stop()

Uncleaned data:


+-----+----+----+
| name| age|city|
+-----+----+----+
|Alice|  25|  NY|
|  Bob|NULL|  LA|
|  Bob|NULL|  LA|
| NULL|  40|  LA|
+-----+----+----+

Removed duplicates:


+-----+----+----+
| name| age|city|
+-----+----+----+
|Alice|  25|  NY|
|  Bob|NULL|  LA|
| NULL|  40|  LA|
+-----+----+----+

Replaced missing ages with average age:
+-----+---+----+
| name|age|city|
+-----+---+----+
|Alice| 25|  NY|
|  Bob| 32|  LA|
| NULL| 40|  LA|
+-----+---+----+

Replaced missing names with 'Unknown':
+-------+---+----+
|   name|age|city|
+-------+---+----+
|  Alice| 25|  NY|
|    Bob| 32|  LA|
|Unknown| 40|  LA|
+-------+---+----+

People over 30:
+-------+---+----+
|   name|age|city|
+-------+---+----+
|    Bob| 32|  LA|
|Unknown| 40|  LA|
+-------+---+----+



In [11]:
# 2. Group By and Aggregation
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("GroupByAndAggregation").master("local[*]").getOrCreate()

salesData = spark.createDataFrame(
    [("A", "Phone", 500),
     ("A", "Laptop", 1000),
     ("B", "Phone", 700)],
    ["customer", "product", "amount"]
)

print("Sales data:")
salesData.show()

# Find the total spent per customer
totalPerCustomer = salesData.groupBy(func.col("customer")).agg(func.sum("amount").alias("total"))
print("Total spent per customer:")
totalPerCustomer.show()

# Find the average purchase amount
averagePurchaseAmount = salesData.agg(func.avg(func.col("amount"))).first()[0]
print(f"Average purchase amount: {averagePurchaseAmount:.2f}")

# Find the highest spending customer (I assume it means whoeveer spent the most in total)
highestSpendingCustomer = totalPerCustomer.orderBy(func.col("total"), ascending=False)
print("Highest spending customer:")
highestSpendingCustomer.show(1)

spark.stop()

Sales data:


+--------+-------+------+
|customer|product|amount|
+--------+-------+------+
|       A|  Phone|   500|
|       A| Laptop|  1000|
|       B|  Phone|   700|
+--------+-------+------+

Total spent per customer:


+--------+-----+
|customer|total|
+--------+-----+
|       A| 1500|
|       B|  700|
+--------+-----+

Average purchase amount: 733.33
Highest spending customer:
+--------+-----+
|customer|total|
+--------+-----+
|       A| 1500|
+--------+-----+
only showing top 1 row


In [12]:
# 3. Top N per Group
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("TopNPerGroup").master("local[*]").getOrCreate()

employeeData = spark.createDataFrame(
    [("IT", "John", 100),
     ("IT", "Mary", 200),
     ("IT", "Alex", 150),
     ("Sales", "Alec", 300), # Added my own records here so that there is more than one department
     ("Sales", "Al", 150),
     ("Sales", "Jeff", 100)],
    ["department", "employee", "salary"]
)

print("Employee data:")
employeeData.show()

# Find the highest-paid employee in each department
ranking = Window.partitionBy(func.col("department")).orderBy(func.desc(func.col("salary")))
# row_number assigns a sequential index to each row
# rank leaves gaps in the sequence if there is a tie
# dense_rank combines the effects of both row_number and rank
departmentRanking = employeeData.withColumn("rank", func.dense_rank().over(ranking))
highestPaidPerDepartment = departmentRanking.where(func.col("rank") == 1)
print("Highest paid employee per department:")
highestPaidPerDepartment.show()

spark.stop()

Employee data:


+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|        IT|    John|   100|
|        IT|    Mary|   200|
|        IT|    Alex|   150|
|     Sales|    Alec|   300|
|     Sales|      Al|   150|
|     Sales|    Jeff|   100|
+----------+--------+------+

Highest paid employee per department:


+----------+--------+------+----+
|department|employee|salary|rank|
+----------+--------+------+----+
|        IT|    Mary|   200|   1|
|     Sales|    Alec|   300|   1|
+----------+--------+------+----+



In [ ]:
# 4. Joins
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("Joins").master("local[*]").getOrCreate()

customers = spark.createDataFrame(
    [(1, "Alice"),
     (2, "Bob")],
    ["id", "name"]
)

print("Customers:")
customers.show()

orders = spark.createDataFrame(
    [(1, 100),
     (1, 50),
     (3, 25)],
    ["customer_id", "amount"]
)

print("Orders:")
orders.show()

# Inner join (default, don't need to specify)
# Dripping the customer_id column to reduce reduncancy
innerJoin = customers.join(orders, func.col("id") == func.col("customer_id")).drop(func.col("customer_id"))
print("Inner join:")
innerJoin.show()

# Left join
leftJoin = customers.join(orders, func.col("id") == func.col("customer_id"), how="left").drop(func.col("customer_id"))
print("Left join:")
leftJoin.show()

# Customers without orders (left anti join)
leftAntiJoin = customers.join(orders, func.col("id") == func.col("customer_id"), how="left_anti").drop(func.col("customer_id"))
print("Customers with no orders:")
leftAntiJoin.show()

# Total spent per customer
# Would it be more efficient to calculate sum on orders and then inner join, or calculate sum on the inner join?
totalPerCustomer = innerJoin.groupBy(func.col("id")).agg(func.sum(func.col("amount")).alias("total"))
#orderSums = orders.groupBy(func.col("customer_id")).agg(func.sum(func.col("amount")).alias("total"))
#totalPerCustomer = customers.join(orderSums, func.col("id") == func.col("customer_id")).drop(func.col("customer_id"))
print("Total spent per customer:")
totalPerCustomer.show()

spark.stop()

Customers:


+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

Orders:
+-----------+------+
|customer_id|amount|
+-----------+------+
|          1|   100|
|          1|    50|
|          3|    25|
+-----------+------+

Inner join:


+---+-----+------+
| id| name|amount|
+---+-----+------+
|  1|Alice|   100|
|  1|Alice|    50|
+---+-----+------+

Left join:


+---+-----+------+
| id| name|amount|
+---+-----+------+
|  1|Alice|    50|
|  1|Alice|   100|
|  2|  Bob|  NULL|
+---+-----+------+

Customers with no orders:


+---+----+
| id|name|
+---+----+
|  2| Bob|
+---+----+

Total spent per customer:
+---+-----+
| id|total|
+---+-----+
|  1|  150|
+---+-----+



'\n4. Joins\n\nCustomers\n\nid\tname\n1\tAlice\n2\tBob\n\nOrders\n\ncustomer_id\tamount\n1\t100\n1\t50\n3\t25\n\nQuestions:\n\nInner join\nLeft join\nCustomers with no orders\nTotal spend per customer\n\nTests:\n\njoins\nnull handling\naggregations\n\n5. Find Duplicate Records\n\nDataset:\n\nemail\n-----\na@test.com\nb@test.com\na@test.com\nc@test.com\n\nTasks:\n\nFind duplicate emails\nCount occurrences\nKeep only unique rows\n\n6. Window Function Challenge\n\nTransactions\n\nuser\tdate\tamount\nA\tJan1\t10\nA\tJan2\t20\nA\tJan3\t15\n\nQuestions:\n\nRunning total\nPrevious transaction\nDifference from previous transaction\nRolling 7-day average\n\nTests:\n\nlag\nlead\ncumulative sums\nwindow frames\n\n7. Sessionization\n\nGiven clickstream events:\n\nuser\ntimestamp\n\nCreate sessions where a new session starts if the gap exceeds 30 minutes.\n\nTests:\n\nwindow functions\ntimestamp arithmetic\ncumulative IDs\n\n8. Pivot Challenge\n\nInput\n\nMonth\tProduct\tSales\nJan\tA\t10\nJan\tB\t

In [2]:
# 5. Find Duplicate Records
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("FindDuplicates").master("local[*]").getOrCreate()

emails = spark.createDataFrame(
    [("a@test.com",),
     ("b@test.com",),
     ("a@test.com",),
     ("c@test.com",)],
    ["email"]
)

print("emails:")
emails.show()

# Find duplicate emails
countOccurences = emails.groupBy(func.col("email")).agg(func.count("email").alias("count"))
duplicateEmails = countOccurences.where(func.col("count") > 1).select(func.col("email"))
print("Duplicate emails:")
duplicateEmails.show()

# Count occurences
print("Count occurences:")
countOccurences.show()

# Keep only unique rows
uniqueRows = emails.dropDuplicates()
print("Unique emails:")
uniqueRows.show()

spark.stop()

emails:


+----------+
|     email|
+----------+
|a@test.com|
|b@test.com|
|a@test.com|
|c@test.com|
+----------+

Duplicate emails:


+----------+
|     email|
+----------+
|a@test.com|
+----------+

Count occurences:
+----------+-----+
|     email|count|
+----------+-----+
|a@test.com|    2|
|b@test.com|    1|
|c@test.com|    1|
+----------+-----+

Unique emails:
+----------+
|     email|
+----------+
|a@test.com|
|b@test.com|
|c@test.com|
+----------+



In [ ]:
# 6. Window Function Challenge
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("WindowFunctions").master("local[*]").getOrCreate()

transactions = spark.createDataFrame(
    [("A", 1, 10),
     ("A", 2, 20),
     ("A", 3, 15),
     ("A", 4, 25),
     ("A", 5, 10),
     ("A", 6, 5),
     ("A", 7, 35),
     ("A", 8, 20),
     ("A", 9, 15)], # Added some more days so that we can actually have a 7-day average
    ["user", "date", "amount"]
)
"""
transactions = spark.createDataFrame(
    [("A", "Jan1", 10),
     ("A", "Jan2", 20),
     ("A", "Jan3", 15),
     ("A", "Jan4", 25),
     ("A", "Jan5", 10),
     ("A", "Jan6", 5),
     ("A", "Jan7", 35),
     ("A", "Jan8", 20),
     ("A", "Jan9", 15)], # Added some more days so that we can actually have a 7-day average
    ["user", "date", "amount"]
)
"""

print("Transactions:")
transactions.show()

# Running total
# Group by user, order by date, running total is sum of all preceding rows including itself
runningWindow = Window.partitionBy(func.col("user")).orderBy(func.col("date")).rowsBetween(Window.unboundedPreceding, Window.currentRow)
runningTotal = transactions.withColumn("running_total", func.sum("amount").over(runningWindow))
print("Running total:")
runningTotal.show()

# Previous transaction
lagWindow = Window.partitionBy(func.col("user")).orderBy(func.col("date"))
previousTransaction = transactions.withColumn(
    "previous",
    func.lag(func.col("amount"), 1).over(lagWindow)
)
print("Preivous transactions:")
previousTransaction.show()

# Difference from previous transaction
# we could just compute a new column using the previousTransaction

previousDifference = previousTransaction.withColumn(
    "difference",
    func.col("amount") - func.col("previous")
)
"""
previousDifference = transactions.withColumn(
    "difference",
    func.col("amount") - func.lag(func.col("amount"), 1).over(lagWindow)
)
"""
print("Difference from previous transation:")
previousDifference.show()

# Rolling 7-day average
rollingWindow = Window.partitionBy(func.col("user")).orderBy(func.col("date")).rangeBetween(-6, 0)
rollingAverage = transactions.withColumn("rolling_average", func.avg("amount").over(rollingWindow))
print("Rolling 7-day average:")
rollingAverage.show()

spark.stop()

Transactions:


+----+----+------+
|user|date|amount|
+----+----+------+
|   A|   1|    10|
|   A|   2|    20|
|   A|   3|    15|
|   A|   4|    25|
|   A|   5|    10|
|   A|   6|     5|
|   A|   7|    35|
|   A|   8|    20|
|   A|   9|    15|
+----+----+------+

Running total:


+----+----+------+-------------+
|user|date|amount|running_total|
+----+----+------+-------------+
|   A|   1|    10|           10|
|   A|   2|    20|           30|
|   A|   3|    15|           45|
|   A|   4|    25|           70|
|   A|   5|    10|           80|
|   A|   6|     5|           85|
|   A|   7|    35|          120|
|   A|   8|    20|          140|
|   A|   9|    15|          155|
+----+----+------+-------------+

Preivous transactions:
+----+----+------+--------+
|user|date|amount|previous|
+----+----+------+--------+
|   A|   1|    10|    NULL|
|   A|   2|    20|      10|
|   A|   3|    15|      20|
|   A|   4|    25|      15|
|   A|   5|    10|      25|
|   A|   6|     5|      10|
|   A|   7|    35|       5|
|   A|   8|    20|      35|
|   A|   9|    15|      20|
+----+----+------+--------+

Difference from previous transation:
+----+----+------+--------+----------+
|user|date|amount|previous|difference|
+----+----+------+--------+----------+
|   A|   1|    10|    NULL| 

In [ ]:
# 7. Sessionization
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as func
from datetime import datetime

sessionName = "Sessionization"
masterString = "local[*]"
spark = SparkSession.builder.appName(sessionName).master(masterString).getOrCreate()

timestamps = spark.createDataFrame(
    [(1, datetime(2026, 7, 9, 12, 0, 0)),
     (1, datetime(2026, 7, 9, 12, 5, 0)),
     (1, datetime(2026, 7, 9, 12, 10, 0)),
     (1, datetime(2026, 7, 9, 12, 15, 0)),
     (1, datetime(2026, 7, 9, 12, 50, 0)),
     (1, datetime(2026, 7, 9, 12, 55, 0)),
     (1, datetime(2026, 7, 9, 1, 0, 0))],
    ["user", "timestamp"]
)

print("Timestamps:")
timestamps.show()

window = Window.partitionBy("user").orderBy(func.col("timestamp"))

timestampsWithPrevious = timestamps.withColumn(
    "previous_timestamp",
    func.lag(func.col("timestamp")).over(window)
)
print("Previous timestamps:")
timestampsWithPrevious.show()

timestampsWithFlag = timestampsWithPrevious.withColumn(
    "is_new_session",
    func.when(
        (func.unix_timestamp(func.col("timestamp")) - func.unix_timestamp(func.col("previous_timestamp"))) / 60 > 30, 1
    ).otherwise(0)
)

timestampSessions = timestampsWithFlag.withColumn(
    "session_id",
    func.sum("is_new_session").over(window)
)
print("Session IDs:")
timestampSessions.show()

spark.stop()

Timestamps:


+----+-------------------+
|user|          timestamp|
+----+-------------------+
|   1|2026-07-09 12:00:00|
|   1|2026-07-09 12:05:00|
|   1|2026-07-09 12:10:00|
|   1|2026-07-09 12:15:00|
|   1|2026-07-09 12:50:00|
|   1|2026-07-09 12:55:00|
|   1|2026-07-09 01:00:00|
+----+-------------------+

Previous timestamps:
+----+-------------------+-------------------+
|user|          timestamp| previous_timestamp|
+----+-------------------+-------------------+
|   1|2026-07-09 01:00:00|               NULL|
|   1|2026-07-09 12:00:00|2026-07-09 01:00:00|
|   1|2026-07-09 12:05:00|2026-07-09 12:00:00|
|   1|2026-07-09 12:10:00|2026-07-09 12:05:00|
|   1|2026-07-09 12:15:00|2026-07-09 12:10:00|
|   1|2026-07-09 12:50:00|2026-07-09 12:15:00|
|   1|2026-07-09 12:55:00|2026-07-09 12:50:00|
+----+-------------------+-------------------+

Session IDs:
+----+-------------------+-------------------+--------------+----------+
|user|          timestamp| previous_timestamp|is_new_session|session_id|
+---

'\n7. Sessionization\n\nGiven clickstream events:\n\nuser\ntimestamp\n\nCreate sessions where a new session starts if the gap exceeds 30 minutes.\n\nTests:\n\nwindow functions\ntimestamp arithmetic\ncumulative IDs\n\n8. Pivot Challenge\n\nInput\n\nMonth\tProduct\tSales\nJan\tA\t10\nJan\tB\t15\nFeb\tA\t12\n\nOutput\n\nMonth\tA\tB\nJan\t10\t15\nFeb\t12\tnull\n\nTests:\n\npivot\n\n9. Explode Nested Data\n\nInput:\n\n[\n  ("Alice", ["Math", "Science"]),\n  ("Bob", ["History"])\n]\n\nOutput:\n\nname\tsubject\nAlice\tMath\nAlice\tScience\nBob\tHistory\n\nTests:\n\narrays\nexplode\n\n10. JSON Parsing\n\nInput:\n\n{\n  "customer":{\n      "id":1,\n      "city":"NY"\n  }\n}\n\nTasks:\n\nParse JSON\nFlatten nested columns\nSelect nested fields\n\nTests:\n\nfrom_json\nschemas\nstructs\n\n11. Skew Handling\n\nQuestion:\n\nOne customer has 50 million transactions while others have only a few hundred.\n\nAsk:\n\nWhy is the job slow?\nHow would you fix it?\n\nExpected discussion:\n\nsalting keys\nbro

In [ ]:
# 8. Pivot Challenge
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("PivotChallenge").master("local[*]").getOrCreate()

monthlyProductSales = spark.createDataFrame(
    [("Jan", "A", 10),
     ("Jan", "B", 15),
     ("Feb", "A", 12)],
    ["month", "product", "sales"]
)

print("Monthly Product Sales:")
monthlyProductSales.show()

# Pivot
# Pivot requires 3 finctions: groupBy(), pivot(), and sum()
#pivotedSales = monthlyProductSales.groupBy(func.col("month")).pivot(func.col("product")).sum(func.col("sales"))
pivotedSales = monthlyProductSales.groupBy("month").pivot("product").sum("sales")
print("Pivoted sales:")
pivotedSales.show()

spark.stop()

Monthly Product Sales:


+-----+-------+-----+
|month|product|sales|
+-----+-------+-----+
|  Jan|      A|   10|
|  Jan|      B|   15|
|  Feb|      A|   12|
+-----+-------+-----+

Pivoted sales:
+-----+---+----+
|month|  A|   B|
+-----+---+----+
|  Feb| 12|NULL|
|  Jan| 10|  15|
+-----+---+----+



'\n8. Pivot Challenge\n\nInput\n\nMonth\tProduct\tSales\nJan\tA\t10\nJan\tB\t15\nFeb\tA\t12\n\nOutput\n\nMonth\tA\tB\nJan\t10\t15\nFeb\t12\tnull\n\nTests:\n\npivot\n\n9. Explode Nested Data\n\nInput:\n\n[\n  ("Alice", ["Math", "Science"]),\n  ("Bob", ["History"])\n]\n\nOutput:\n\nname\tsubject\nAlice\tMath\nAlice\tScience\nBob\tHistory\n\nTests:\n\narrays\nexplode\n\n10. JSON Parsing\n\nInput:\n\n{\n  "customer":{\n      "id":1,\n      "city":"NY"\n  }\n}\n\nTasks:\n\nParse JSON\nFlatten nested columns\nSelect nested fields\n\nTests:\n\nfrom_json\nschemas\nstructs\n\n11. Skew Handling\n\nQuestion:\n\nOne customer has 50 million transactions while others have only a few hundred.\n\nAsk:\n\nWhy is the job slow?\nHow would you fix it?\n\nExpected discussion:\n\nsalting keys\nbroadcast joins\nrepartitioning\nadaptive query execution\nskew join optimization\n\n13. Optimize a Slow Job\n\nGiven code like:\n\ndf.join(df2, "id").groupBy("city").count()\n\nAsk:\n\nHow would you optimize this?\n\

In [ ]:
# 9. Explode Nested Data
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("ExplodeNestedData").master("local[*]").getOrCreate()

nestedData = spark.createDataFrame(
    [("Alice", ["Math", "Science"]),
     ("Bob", ["History"])],
    ["name", "courses"]
)

print("Nested Data:")
nestedData.show()

# Explode nested data
explodedData = nestedData.select(func.col("name"), func.explode(func.col("courses")).alias("subject"))
print("Exploded Data:")
explodedData.show()

spark.stop()

Nested Data:


+-----+---------------+
| name|        courses|
+-----+---------------+
|Alice|[Math, Science]|
|  Bob|      [History]|
+-----+---------------+

Exploded Data:
+-----+-------+
| name|subject|
+-----+-------+
|Alice|   Math|
|Alice|Science|
|  Bob|History|
+-----+-------+



'\n9. Explode Nested Data\n\nInput:\n\n[\n  ("Alice", ["Math", "Science"]),\n  ("Bob", ["History"])\n]\n\nOutput:\n\nname\tsubject\nAlice\tMath\nAlice\tScience\nBob\tHistory\n\nTests:\n\narrays\nexplode\n\n10. JSON Parsing\n\nInput:\n\n{\n  "customer":{\n      "id":1,\n      "city":"NY"\n  }\n}\n\nTasks:\n\nParse JSON\nFlatten nested columns\nSelect nested fields\n\nTests:\n\nfrom_json\nschemas\nstructs\n\n11. Skew Handling\n\nQuestion:\n\nOne customer has 50 million transactions while others have only a few hundred.\n\nAsk:\n\nWhy is the job slow?\nHow would you fix it?\n\nExpected discussion:\n\nsalting keys\nbroadcast joins\nrepartitioning\nadaptive query execution\nskew join optimization\n\n13. Optimize a Slow Job\n\nGiven code like:\n\ndf.join(df2, "id").groupBy("city").count()\n\nAsk:\n\nHow would you optimize this?\n\nPossible discussion:\n\nbroadcast joins\ncaching\npartition pruning\npredicate pushdown\navoiding unnecessary shuffles\nchoosing appropriate partition counts\n\n14

In [ ]:
# 10. JSON parsing
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName("JSONParsing").master("local[*]").getOrCreate()

jsstr = '{"customer": {"id": 1, "city": "NY"}}'

# Since the json is nested, our schema needs to be too
innerSchema = StructType([
    StructField("id", IntegerType()),
    StructField("city", StringType())
])
schema = StructType([
    StructField("customer", innerSchema)
])

# Parse JSON
# Read unparsed into DF first
data = [(1, jsstr)]
unparsed = spark.createDataFrame(data, ["temp_id", "json_data"])
print("Unparsed:")
unparsed.show()
# Parse it
parsedAndNestedTwice = unparsed.withColumn("parsed", func.from_json(func.col("json_data"), schema))
print("Parsed:")
parsedAndNestedTwice.show()
# Un-nest it (twice)
parsedAndNestedOnce = parsedAndNestedTwice.select(func.col("parsed.*"))
parsed = parsedAndNestedOnce.select(func.col("customer.id"), func.col("customer.city")) # Being explicit here to control print order
print("Un-nested:")
parsed.show()

spark.stop()

Unparsed:


+-------+--------------------+
|temp_id|           json_data|
+-------+--------------------+
|      1|{"customer": {"id...|
+-------+--------------------+

Parsed:
+-------+--------------------+---------+
|temp_id|           json_data|   parsed|
+-------+--------------------+---------+
|      1|{"customer": {"id...|{{1, NY}}|
+-------+--------------------+---------+

Un-nested:
+---+----+
| id|city|
+---+----+
|  1|  NY|
+---+----+



'\n11. Skew Handling\n\nQuestion:\n\nOne customer has 50 million transactions while others have only a few hundred.\n\nAsk:\n\nWhy is the job slow?\nHow would you fix it?\n\nExpected discussion:\n\nsalting keys\nbroadcast joins\nrepartitioning\nadaptive query execution\nskew join optimization\n\n13. Optimize a Slow Job\n\nGiven code like:\n\ndf.join(df2, "id").groupBy("city").count()\n\nAsk:\n\nHow would you optimize this?\n\nPossible discussion:\n\nbroadcast joins\ncaching\npartition pruning\npredicate pushdown\navoiding unnecessary shuffles\nchoosing appropriate partition counts\n\n14. Read Multiple Files\n\nTasks:\n\nRead all CSVs from a directory\nInfer schema (or define it explicitly)\nAdd filename as a column\nHandle malformed records\n\nTests:\n\nfile ingestion\nschema management\nmetadata columns\n\n15. End-to-End ETL Challenge (Excellent 60-minute Interview)\n\nInput:\n\nOrders:\n\norder_id\ncustomer_id\ndate\namount\n\nCustomers:\n\ncustomer_id\ncity\nsegment\n\nRequirements:


11. Skew Handling

Question:

One customer has 50 million transactions while others have only a few hundred.

Ask:

Why is the job slow?
How would you fix it?

Expected discussion:

salting keys
broadcast joins
repartitioning
adaptive query execution
skew join optimization

Response:

The reason this job takes so long is the uneven distribution of work across the executors. Both customers here get the same number of executors even though they have different workloads. As a result, the customer with the smaller workload will finish first, and their executors will then have to sit idle while they wait for the other customer to finish their 50 million transactions, reducing the parallelism severely. To remedy this, I would repartition the work so that each executor gets an (approximately) equal share of the workload, to ensure we don't leave any nodes sitting idle for very long.

13. Optimize a Slow Job

Given code like:

df.join(df2, "id").groupBy("city").count()

Ask:

How would you optimize this?

Possible discussion:

broadcast joins
caching
partition pruning
predicate pushdown
avoiding unnecessary shuffles
choosing appropriate partition counts

Response:

The main optimization that comes to mind is to perform the group by before the join if possible. Since groupby triggers a full shuffle, grouping before a join helps minimize the amount of data the system has to shuffle around. Another possibility, if the dataset isn't too terribly large compared to executor momory, is to perform a broadcast join across all executors, which eliminates the need for a shuffle entirely. Lastly, we could adjust the number of partitions to reduce the effects of a full shuffle.

14. Read Multiple Files

Tasks:

Read all CSVs from a directory
Infer schema (or define it explicitly)
Add filename as a column
Handle malformed records

Example code:
df = spark.read.csv("/directory/").option("inferschema", True)
named_df = df.withColumn("filename", input_file_name())
handled_df = named_df.fillna("replacement value", subset=[columns])

Tests:

file ingestion
schema management
metadata columns

In [ ]:
# 15. End-to-End ETL Challenge
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("ETL").master("local[*]").getOrCreate()

orders = spark.createDataFrame(
    [(1, 1, "Jan1", 100),
     (2, 1, "Jan2", 200),
     (3, 1, "Jan3", None),
     (4, 2, None, 300),
     (5, 2, "Jan5", 200),
     (6, 2, "Jan7", 200),
     (7, 3, "Dec30", 100),
     (8, 3, "Jan3", 20),
     (9, 3, "Jan8", 10),
     (10, 4, "Mar9", 300),
     (11, 4, "Apr5", 200),
     (12, 4, "May7", 800)],
    ["order_id", "customer_id", "date", "amount"]
)
print("Orders:")
orders.show()

customers = spark.createDataFrame(
    [(1, "Dayton", "IDK what segment means here!"),
     (2, "Columbus", "Seriously I don't, what did the AI mean by it?"),
     (3, "Dayton", "Seriously I don't, what did the AI mean by it?"),
     (4, "Columbus", "UMM")],
    ["customer_id", "city", "segment"]
)
print("Customers:")
customers.show()

# Clean missing values
# using 100 as default amount
# Getting rid of entries with no valid time
correctedAmount = orders.fillna(100, subset="amount")
cleanedOrders = correctedAmount.where(func.col("date").isNotNull())
print("Cleaned Orders:")
cleanedOrders.show()

# Remove duplicates
uniqueOrders = cleanedOrders.dropDuplicates()
print("Unique Orders:")
uniqueOrders.show()

# Join datasets
customersAndOrders = customers.join(uniqueOrders, on="customer_id").drop(func.col("orders.customer_id")).cache()
print("Customers + Orders:")
customersAndOrders.show()

# Total sales by city
salesByCity = customersAndOrders.groupBy(func.col("city")).agg(func.sum(func.col("amount")).alias("total_sales"))
print("Total sales by city:")
salesByCity.show()

# Average order value by segment
averageOrderValueBySegment = customersAndOrders.groupBy(func.col("segment")).agg(func.avg(func.col("amount")).alias("avg_amount"))
print("Average order value by segment:")
averageOrderValueBySegment.show()

# Top 3 customers by spending (does this expect us to use window functions?)
topCustomers = (
    customersAndOrders.groupBy(func.col("customer_id"))
    .agg(func.sum(func.col("amount")).alias("total_spending"))
    .orderBy(func.col("total_spending"), ascending=False)
)
print("Top 3 customers by spending:")
topCustomers.show(3)

# Write the result partitioned by city (what exactly am I writing?)
partitionedJoin = customersAndOrders.repartition(5, func.col("city"))
partitionedJoin.write.csv("./customerJoinResults")

spark.stop()

Orders:


+--------+-----------+-----+------+
|order_id|customer_id| date|amount|
+--------+-----------+-----+------+
|       1|          1| Jan1|   100|
|       2|          1| Jan2|   200|
|       3|          1| Jan3|  NULL|
|       4|          2| NULL|   300|
|       5|          2| Jan5|   200|
|       6|          2| Jan7|   200|
|       7|          3|Dec30|   100|
|       8|          3| Jan3|    20|
|       9|          3| Jan8|    10|
|      10|          4| Mar9|   300|
|      11|          4| Apr5|   200|
|      12|          4| May7|   800|
+--------+-----------+-----+------+

Customers:
+-----------+--------+--------------------+
|customer_id|    city|             segment|
+-----------+--------+--------------------+
|          1|  Dayton|IDK what segment ...|
|          2|Columbus|Seriously I don't...|
|          3|  Dayton|Seriously I don't...|
|          4|Columbus|                 UMM|
+-----------+--------+--------------------+

Cleaned Orders:
+--------+-----------+-----+------+
|order_

+-----------+--------+--------------------+--------+-----+------+
|customer_id|    city|             segment|order_id| date|amount|
+-----------+--------+--------------------+--------+-----+------+
|          1|  Dayton|IDK what segment ...|       3| Jan3|   100|
|          1|  Dayton|IDK what segment ...|       2| Jan2|   200|
|          1|  Dayton|IDK what segment ...|       1| Jan1|   100|
|          3|  Dayton|Seriously I don't...|       9| Jan8|    10|
|          3|  Dayton|Seriously I don't...|       8| Jan3|    20|
|          3|  Dayton|Seriously I don't...|       7|Dec30|   100|
|          2|Columbus|Seriously I don't...|       6| Jan7|   200|
|          2|Columbus|Seriously I don't...|       5| Jan5|   200|
|          4|Columbus|                 UMM|      12| May7|   800|
|          4|Columbus|                 UMM|      11| Apr5|   200|
|          4|Columbus|                 UMM|      10| Mar9|   300|
+-----------+--------+--------------------+--------+-----+------+

Total sal

+--------+-----------+
|    city|total_sales|
+--------+-----------+
|  Dayton|        530|
|Columbus|       1700|
+--------+-----------+

Average order value by segment:


+--------------------+------------------+
|             segment|        avg_amount|
+--------------------+------------------+
|IDK what segment ...|133.33333333333334|
|Seriously I don't...|             106.0|
|                 UMM| 433.3333333333333|
+--------------------+------------------+

Top 3 customers by spending:


+-----------+--------------+
|customer_id|total_spending|
+-----------+--------------+
|          4|          1300|
|          1|           400|
|          2|           400|
+-----------+--------------+
only showing top 3 rows


"\n15. End-to-End ETL Challenge (Excellent 60-minute Interview)\n\nInput:\n\nOrders:\n\norder_id\ncustomer_id\ndate\namount\n\nCustomers:\n\ncustomer_id\ncity\nsegment\n\nRequirements:\n\nClean missing values\nRemove duplicates\nJoin datasets\nCalculate:\ntotal sales by city\naverage order value by segment\ntop 3 customers by spending\nWrite the result partitioned by city\n\nTests nearly every core PySpark skill:\n\nreading data\njoins\naggregations\nwindow functions\npartitioned output\n\nCommon Follow-Up Questions\n\nAfter a coding exercise, interviewers often ask conceptual questions such as:\n\nWhat causes a shuffle, and why is it expensive?\nWhen would you use cache() versus persist()?\nHow do narrow and wide transformations differ?\nWhen should you use a broadcast join?\nHow do coalesce() and repartition() differ?\nHow do you diagnose a slow Spark job?\nWhat are stages, tasks, and partitions in Spark?\nHow does Spark's lazy evaluation affect execution?\n"

Common Follow-Up Questions

After a coding exercise, interviewers often ask conceptual questions such as:

What causes a shuffle, and why is it expensive?

Shuffling is caused by any operation that changes how the data is organized such as join(), groupBy(), orderBy(), or repartition(). Shuffling is a very expensive operation since it involves heavy I/O on the network, CPU, and disk.

When would you use cache() versus persist()?

Cache() is a shorthand for persist() to store data to main memory. Data in memory can be accessed much faster than accessing from disk, so cache() is preferred when we need to access the data again quickly. However, since main memory is lost in the event of a crash, using persist() to write to disk would make the program more resistant to node failure.

How do narrow and wide transformations differ?

Narrow transformations act on a row-by-row basis, and don't change the general organization of the data. Wide transformations, however, change how the data is organized among partitions, and therefore require shuffling.

When should you use a broadcast join?
How do coalesce() and repartition() differ?

Repartition() is capable of both increasing and decreasing the number of partitions, while coalesce() can only decrease. Additionally, repartition() shuffles and redistributes data evenly across the new set of partitions. Coalesce(), by contrast, simply merges cut partition data onto nearby partitions, avoiding shuffle entirely. However, this usually creates data skew, since some partitions will end up with more data than others.

How do you diagnose a slow Spark job?
What are stages, tasks, and partitions in Spark?

Stages are collections of transformations that can be executed together without moving data across the network (a.k.a. shuffling). Shuffling divides a job into different stages.

Tasks are the smallest unit of work in spark. It represents a single thread of execution running stage logic on a single partition. Stages are divided into tasks based on the number of executors.

Partitions are chunks of an RDD or dataframe. The more partitions a dataset has, the more parallel it executes.

How does Spark's lazy evaluation affect execution?

Lazy evaluation means that commands don't run unless the value they compute is actually called for. This ensures that values aren't calculated unless theyh are actually needed, and allows the optimizer to determine the most efficient way to calculate them.